## Exercise — What a Regression Model Is

### Task 1 — Build the examples a model learns from

A supervised regression model learns its rule from examples where the answer is already known from the lab. Build that table.

Here is the data from the video:

```py
compounds = {
    "name": ["aspirin", "caffeine", "paracetamol", "ibuprofen", "salicylic acid",
             "naproxen", "diazepam", "benzene", "phenol", "toluene"],
    "smiles": ["CC(=O)Oc1ccccc1C(=O)O", "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "CC(=O)Nc1ccc(O)cc1",
               "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "OC(=O)c1ccccc1O", "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
               "CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc12", "c1ccccc1", "Oc1ccccc1", "Cc1ccccc1"],
    "logS": [-1.72, -0.87, -1.03, -3.97, -1.82, -4.16, -3.75, -1.64, 0.00, -2.21],
}
```

Turn it into a DataFrame called `df`. Check that every SMILES parses, and print a warning for any that does not. Then add three columns:

- `mol`, holding the molecule objects
- `mw`, the molecular weight from `Descriptors.MolWt`
- `logp`, the calculated logP from `Descriptors.MolLogP`

Print the columns `name`, `mw`, `logp` and `logS`, rounded to 2 decimals. Each row should show the computed features alongside the measured number that the rule will learn to reproduce.


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
compounds = {
    "name": ["aspirin", "caffeine", "paracetamol", "ibuprofen", "salicylic acid",
             "naproxen", "diazepam", "benzene", "phenol", "toluene"],
    "smiles": ["CC(=O)Oc1ccccc1C(=O)O", "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "CC(=O)Nc1ccc(O)cc1",
               "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "OC(=O)c1ccccc1O", "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
               "CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc12", "c1ccccc1", "Oc1ccccc1", "Cc1ccccc1"],
    "logS": [-1.72, -0.87, -1.03, -3.97, -1.82, -4.16, -3.75, -1.64, 0.00, -2.21],
}
df = pd.DataFrame(compounds)
for name, smiles in zip(df["name"], df["smiles"]):
    if Chem.MolFromSmiles(smiles) is None:
        print("invalid SMILES:", name, smiles)

df["mol"] = [Chem.MolFromSmiles(s) for s in df["smiles"]]
df["mw"] = [Descriptors.MolWt(m) for m in df["mol"]]
df["logp"] = [Descriptors.MolLogP(m) for m in df["mol"]]
print(df[["name", "mw", "logp", "logS"]].round(2))

### Task 2 — Features in, target out

A colleague tried to separate the features from the target. They wrote:

```py
X = df["mw", "logp"]
y = df["logS"]
```

The first line fails with a `KeyError`, because pandas looks for a single column literally named `("mw", "logp")`.

Correct the feature line so that `X` holds the two descriptor columns `mw` and `logp`. Keep `y` as the target column `logS`. Then print the shape of `X` and the shape of `y`.

You should see 10 rows by 2 columns for `X`, and 10 values for `y`.


In [ ]:
X = df[["mw", "logp"]]
y = df["logS"]
print("features:", X.shape)
print("target:", y.shape)

### Task 3 — Predict, then check: the train/test split

Read these lines. Before running anything, write down what the first, second and fourth `print` will show:

```py
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)
print("training rows:", train.shape[0])
print("test rows:", test.shape[0])
print("test compounds:", list(test["name"]))
train_again = df.sample(frac=0.8, random_state=42)
print("same split:", list(train.index) == list(train_again.index))
```

Now run exactly these lines and compare the output with your predictions. Note which two compounds end up in the test set. You will need `train` and `test` in the next task.


In [ ]:
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)
print("training rows:", train.shape[0])
print("test rows:", test.shape[0])
print("test compounds:", list(test["name"]))
train_again = df.sample(frac=0.8, random_state=42)
print("same split:", list(train.index) == list(train_again.index))

### Task 4 — Predict, then check: a model that only memorises

This is the lazy "model" from the video. It is a dictionary that remembers the log S of every training compound by name:

```py
memory = dict(zip(train["name"], train["logS"]))
train_errors = [abs(memory[n] - v) for n, v in zip(train["name"], train["logS"])]
print("total training error:", sum(train_errors))
for name in test["name"]:
    print(name, "->", memory.get(name, "no answer"))
```

Before running, predict two things:

1. What total training error does it print?
2. What does it print for each test compound?

Then run the lines and check your predictions. Compare the perfect training score with the result on the test set. That difference is why we judge a model only on molecules it has never seen.


In [ ]:
memory = dict(zip(train["name"], train["logS"]))
train_errors = [abs(memory[n] - v) for n, v in zip(train["name"], train["logS"])]
print("total training error:", sum(train_errors))
for name in test["name"]:
    print(name, "->", memory.get(name, "no answer"))

### Task 5 — Three targets worth predicting

Build a dictionary called `targets` with one entry for each of the three properties from the video that are worth predicting.

- Each key is the property, for example `"aqueous solubility"`.
- Each value is a short string saying what the target column holds, including the quantity and its units or conditions.

Loop over `targets.items()` and print one line per property, in the form `property: what the target column holds`.


In [ ]:
targets = {
    "aqueous solubility": "log S, log10 of solubility in mol/L",
    "lipophilicity": "measured logD at pH 7.4",
    "potency": "pIC50, -log10 of IC50 in M",
}
for prop, column in targets.items():
    print(f"{prop}: {column}")